# Notebook 01: Train + Generate (DCC26)

This notebook covers the second workshop segment (30 minutes):

1. Build a lightweight conditional generator
2. Train on a small subset for deterministic runtime
3. Generate designs from sampled conditions
4. Save artifacts for Notebook 02

Fallback options are included if you skip training.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'torch', 'torchvision', 'matplotlib', 'pandas']

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import torch as th
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from engibench.problems.beams2d.v0 import Beams2D

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
if th.cuda.is_available():
    th.cuda.manual_seed_all(SEED)

DEVICE = th.device('cuda' if th.cuda.is_available() else 'cpu')
print('device:', DEVICE)

ARTIFACT_DIR = Path('workshops/dcc26/artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = ARTIFACT_DIR / 'mini_cond_generator.pt'


In [ ]:
problem = Beams2D(seed=SEED)
train_ds = problem.dataset['train']
test_ds = problem.dataset['test']

condition_keys = problem.conditions_keys
print('condition keys:', condition_keys)

# Build compact train subset to keep runtime stable in workshop
N_TRAIN = 512
subset_idx = np.random.default_rng(SEED).choice(len(train_ds), size=N_TRAIN, replace=False)

conds_np = np.stack([np.array(train_ds[k])[subset_idx].astype(np.float32) for k in condition_keys], axis=1)
designs_np = np.array(train_ds['optimal_design'])[subset_idx].astype(np.float32)

# Downsample target to reduce model output size and speed up training
designs_t = th.tensor(designs_np).unsqueeze(1)
lowres_t = F.interpolate(designs_t, size=(25, 50), mode='bilinear', align_corners=False).squeeze(1)
targets_np = lowres_t.reshape(N_TRAIN, -1).numpy()

print('conditions shape:', conds_np.shape)
print('designs shape:', designs_np.shape)
print('lowres target shape:', targets_np.shape)

In [ ]:
class MiniCondGenerator(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, out_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


def upsample_to_design(y_flat: th.Tensor) -> th.Tensor:
    low = y_flat.reshape(-1, 1, 25, 50)
    high = F.interpolate(low, size=(50, 100), mode='bilinear', align_corners=False)
    return high.squeeze(1)


model = MiniCondGenerator(in_dim=conds_np.shape[1], out_dim=targets_np.shape[1]).to(DEVICE)
optimizer = th.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

In [ ]:
TRAIN_FROM_SCRATCH = True
EPOCHS = 8
BATCH_SIZE = 64

if TRAIN_FROM_SCRATCH:
    ds = TensorDataset(th.tensor(conds_np), th.tensor(targets_np))
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for xb, yb in dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.item())

        print(f'epoch {epoch + 1:02d}/{EPOCHS} - loss: {epoch_loss / len(dl):.4f}')

    th.save({'model': model.state_dict(), 'condition_keys': condition_keys}, CKPT_PATH)
    print('saved checkpoint to', CKPT_PATH)
elif CKPT_PATH.exists():
    ckpt = th.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print('loaded checkpoint from', CKPT_PATH)
else:
    print('No checkpoint found. Use fallback generation cell below.')

In [ ]:
# Optional fallback: nearest-neighbor by condition vector
USE_NEAREST_NEIGHBOR_FALLBACK = False

rng = np.random.default_rng(SEED)
N_SAMPLES = 24
selected = rng.choice(len(test_ds), size=N_SAMPLES, replace=False)

test_conds = np.stack([np.array(test_ds[k])[selected].astype(np.float32) for k in condition_keys], axis=1)
baseline_designs = np.array(test_ds['optimal_design'])[selected].astype(np.float32)

if USE_NEAREST_NEIGHBOR_FALLBACK:
    generated = []
    for c in test_conds:
        dists = np.linalg.norm(conds_np - c[None, :], axis=1)
        idx = int(np.argmin(dists))
        generated.append(designs_np[idx])
    gen_designs = np.array(generated, dtype=np.float32)
else:
    model.eval()
    with th.no_grad():
        pred_low = model(th.tensor(test_conds, device=DEVICE))
        gen_designs_t = upsample_to_design(pred_low).clamp(0.0, 1.0)
    gen_designs = gen_designs_t.detach().cpu().numpy().astype(np.float32)

print('generated shape:', gen_designs.shape)
print('baseline shape:', baseline_designs.shape)

In [ ]:
conditions_records = []
for i in range(N_SAMPLES):
    rec = {}
    for j, k in enumerate(condition_keys):
        v = test_conds[i, j]
        rec[k] = bool(v) if k == 'overhang_constraint' else float(v)
    conditions_records.append(rec)

np.save(ARTIFACT_DIR / 'generated_designs.npy', gen_designs)
np.save(ARTIFACT_DIR / 'baseline_designs.npy', baseline_designs)
with open(ARTIFACT_DIR / 'conditions.json', 'w', encoding='utf-8') as f:
    json.dump(conditions_records, f, indent=2)

print('Saved artifacts to', ARTIFACT_DIR)

In [ ]:
# Quick visual check of generated designs
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(gen_designs[i], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    ax.set_title(f'gen {i}')
plt.tight_layout()
plt.show()

## Next

Continue with **Notebook 02** to validate and evaluate generated designs against baselines.
